# Model Training and Evaluation

This notebook trains Logistic Regression and Random Forest models using the project's prepared train/test split. It logs each run to local MLflow, compares test metrics,
and saves the selected model to `models/best_model.joblib`.

Run from the repository environment after preprocessing has created `data/processed/`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import mlflow
import mlflow.sklearn

# Find the repository root whether Jupyter starts here or in notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("Open this notebook from the repository or its notebooks folder.")
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.train import (
    load_processed_data,
    train_logistic_regression,
    train_random_forest,
    evaluate_model,
    plot_confusion_matrix,
)

DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"
SCREENSHOTS_DIR = PROJECT_ROOT / "screenshots"
MODEL_DIR.mkdir(exist_ok=True)
SCREENSHOTS_DIR.mkdir(exist_ok=True)
mlflow.set_tracking_uri((PROJECT_ROOT / "mlruns").resolve().as_uri())
mlflow.set_experiment("Heart Disease Prediction")
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
X_train, X_test, y_train, y_test = load_processed_data(DATA_DIR)
print(f"Training rows: {len(X_train)} | Test rows: {len(X_test)}")
print(f"Features: {list(X_train.columns)}")
assert X_train.shape[1] == X_test.shape[1]
assert set(y_train.unique()).issubset({0, 1})

## Train Logistic Regression

The project helper performs cross-validated hyperparameter search. The test set is
used only for final evaluation.

In [ ]:
with mlflow.start_run(run_name="Logistic Regression notebook"):
    lr_model, lr_params = train_logistic_regression(X_train, y_train)
    lr_metrics, lr_pred, lr_pred_proba = evaluate_model(
        lr_model, X_test, y_test, "Logistic Regression"
    )
    mlflow.log_params(lr_params)
    mlflow.log_metrics({f"test_{name}": value for name, value in lr_metrics.items()})
    mlflow.sklearn.log_model(lr_model, artifact_path="model")
print("Best Logistic Regression parameters:", lr_params)
print("Test metrics:", lr_metrics)

## Train Random Forest

This model uses the same prepared split and evaluation metrics for a fair comparison.

In [ ]:
with mlflow.start_run(run_name="Random Forest notebook"):
    rf_model, rf_params = train_random_forest(X_train, y_train)
    rf_metrics, rf_pred, rf_pred_proba = evaluate_model(
        rf_model, X_test, y_test, "Random Forest"
    )
    mlflow.log_params(rf_params)
    mlflow.log_metrics({f"test_{name}": value for name, value in rf_metrics.items()})
    mlflow.sklearn.log_model(rf_model, artifact_path="model")
print("Best Random Forest parameters:", rf_params)
print("Test metrics:", rf_metrics)

## Compare models and save artifacts

The model with the higher test ROC-AUC is selected as the API's `best_model.joblib`.

In [ ]:
results = pd.DataFrame(
    [lr_metrics, rf_metrics],
    index=["Logistic Regression", "Random Forest"],
).rename_axis("model")
display(results.style.format("{:.3f}"))

best_name, best_model = (
    ("Random Forest", rf_model)
    if rf_metrics["roc_auc"] > lr_metrics["roc_auc"]
    else ("Logistic Regression", lr_model)
)

import joblib
joblib.dump(lr_model, MODEL_DIR / "logistic_regression.joblib")
joblib.dump(rf_model, MODEL_DIR / "random_forest.joblib")
joblib.dump(best_model, MODEL_DIR / "best_model.joblib")
print(f"Selected model: {best_name}")
print(f"Saved best model: {MODEL_DIR / 'best_model.joblib'}")

In [ ]:
plot_confusion_matrix(
    y_test, lr_pred, "Logistic Regression",
    SCREENSHOTS_DIR / "05_lr_confusion_matrix.png",
)
plot_confusion_matrix(
    y_test, rf_pred, "Random Forest",
    SCREENSHOTS_DIR / "06_rf_confusion_matrix.png",
)

## Reproducibility notes

The notebook uses the saved train/test split and the same helper functions as
`src/models/train.py`. MLflow run metadata is written under the repository's `mlruns/`
directory. Re-run the preprocessing pipeline first if `data/processed/` is missing.